In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Test UPLINK**

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import scipy.stats as stats

# ==============================================================================
# 0. ESTILO DE TESIS (CONFIGURACIÓN MAESTRA)
# ==============================================================================
def configurar_estilo_tesis():
    sns.set_theme(style="whitegrid", context="paper")
    plt.rcParams.update({
        'font.family': 'serif',
        'axes.labelsize': 11,
        'axes.titlesize': 12,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'figure.titlesize': 14,
        'grid.alpha': 0.3,
        'grid.linestyle': '--',
        'lines.linewidth': 2,
    })

# Color Oficial XGBoost (Carmesí)
COLOR_XGB = '#900C3F'

configurar_estilo_tesis()

# ==============================================================================
# 1. CONFIGURACIÓN
# ==============================================================================
DIR_DATA   = '/content/drive/MyDrive/TESIS 100%/Datos_Entrenamiento_Finales/UpLink/'
DIR_MODELO = '/content/drive/MyDrive/TESIS 100%/XGBoost/Uplink/Entrenamiento/'
DIR_SALIDA = '/content/drive/MyDrive/TESIS 100%/XGBoost/Uplink/Prueba/Test_Final_Uniforme/'

if not os.path.exists(DIR_SALIDA): os.makedirs(DIR_SALIDA)

# Constantes Físicas
FREQ_MHZ = 915.0
H_GATEWAY_EMPIRICOS = 30.0
H_NODO_EMPIRICOS = 1.5
MIN_DISTANCIA = 200
BIN_SIZE = 200

print("🚀 INICIANDO TEST GENERAL UPLINK - XGBOOST (ESTILO UNIFICADO)...")

# ==============================================================================
# 2. CARGA DE DATOS Y MODELO (BLINDADO)
# ==============================================================================
try:
    test_df = pd.read_csv(os.path.join(DIR_DATA, 'Test_UpLink.csv'))

    # Carga Modelo
    ruta_modelo = os.path.join(DIR_MODELO, 'Modelo_XGBoost_Optimizado.pkl')
    if not os.path.exists(ruta_modelo):
        ruta_modelo = os.path.join(DIR_MODELO, 'Modelo_XGBoost_Uplink.pkl')

    if not os.path.exists(ruta_modelo):
         raise FileNotFoundError(f"No se encuentra el modelo en: {DIR_MODELO}")

    xgb_model = joblib.load(ruta_modelo)
    print(f"✅ Modelo cargado: {os.path.basename(ruta_modelo)}")

except Exception as e:
    print(f"❌ Error Fatal Carga: {e}"); exit()

# --- CORRECCIÓN INTELIGENTE DE COLUMNAS ---

# 1. SF (Spreading Factor)
posibles_sf = ['SpreedFactor', 'SpreedFactor_UpLink', 'SpreadingFactor', 'SF']
col_sf = next((c for c in posibles_sf if c in test_df.columns), None)

if not col_sf:
    print("❌ Error: No se encontró columna SF."); exit()

# Alineación con el modelo
if hasattr(xgb_model, 'feature_names_in_'):
    feats_model = list(xgb_model.feature_names_in_)
    sf_model = next((f for f in feats_model if 'SpreedFactor' in f or 'SF' in f), None)

    if sf_model and col_sf != sf_model:
        print(f"   ℹ️ Renombrando '{col_sf}' -> '{sf_model}' para el modelo.")
        test_df.rename(columns={col_sf: sf_model}, inplace=True)
        col_sf = sf_model

# 2. Target
posibles_target = ['Path_Loss_UpLink', 'Path_Loss_Uplink', 'Path_Loss']
TARGET = next((c for c in posibles_target if c in test_df.columns), None)
if not TARGET: print("❌ Error: No se encontró Target."); exit()

# 3. Definición Final de Features
if hasattr(xgb_model, 'feature_names_in_'):
    FEATURES = list(xgb_model.feature_names_in_)
else:
    FEATURES = ['Distance', 'Gateway_Altitude', 'EndDevice_Altitude', col_sf, 'Terreno', 'Power_UpLink']

print(f"   ℹ️ Features usados: {FEATURES}")

for c in FEATURES + [TARGET]:
    if c not in test_df.columns: print(f"❌ Falta columna requerida: {c}"); exit()

# Filtro Distancia
test_df = test_df[test_df['Distance'] >= MIN_DISTANCIA].copy()

# ==============================================================================
# 3. CÁLCULO DE PREDICCIONES
# ==============================================================================
print("📐 Calculando predicciones...")

# A. XGBoost
try:
    test_df['Pred_XGB'] = xgb_model.predict(test_df[FEATURES])
except Exception as e:
    print(f"❌ Error predicción XGB: {e}"); exit()

# B. Empíricos
d_km = test_df['Distance'] / 1000.0; d_m = test_df['Distance']
h_b = H_GATEWAY_EMPIRICOS; h_m = H_NODO_EMPIRICOS; f = FREQ_MHZ

test_df['Pred_FSPL'] = 32.44 + 20*np.log10(f) + 20*np.log10(d_km)
g_f = 44.49*np.log10(f) - 4.78*(np.log10(f))**2
test_df['Pred_Ericsson'] = (36.2 + 30.2*np.log10(d_km) - 12.0*np.log10(h_b) + 0.1*np.log10(h_b)*np.log10(d_km) - 3.2*(np.log10(11.75*h_m))**2 + g_f)
lam = 300.0 / f
A_sui = 20 * np.log10((4 * np.pi * 100) / lam)
gamma_sui = 4.0 - 0.0065 * h_b + 17.1 / h_b
test_df['Pred_SUI'] = A_sui + 10 * gamma_sui * np.log10(d_m / 100.0) + 6.0 * np.log10(f / 2000.0) - 10.8 * np.log10(h_m / 2.0)
test_df['Pred_Egli'] = 88.0 + 20*np.log10(f) + 40*np.log10(d_km) - 20*np.log10(h_b) - 10*np.log10(h_m)

# FI y LNS
x_log = 10 * np.log10(d_m)
slope_fi, intercept_fi, _, _, _ = linregress(x_log, test_df[TARGET])
test_df['Pred_FI'] = intercept_fi + slope_fi * x_log

fspl_d0 = 32.44 + 20*np.log10(f)
y_diff = test_df[TARGET] - fspl_d0
x_term = 10 * np.log10(d_km)
n_opt = np.sum(x_term * y_diff) / np.sum(x_term**2)
test_df['Pred_LNS'] = fspl_d0 + n_opt * x_term

# ==============================================================================
# 4. MÉTRICAS GLOBALES
# ==============================================================================
def calc_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {'RMSE': np.sqrt(mse), 'MSE': mse, 'MAE': mean_absolute_error(y_true, y_pred), 'R2': r2_score(y_true, y_pred)}

nombres = ['XGBoost', 'Floating Intercept', 'Log-Normal Shad.', 'SUI', 'Ericsson', 'Egli', 'FSPL']
cols_res = ['Pred_XGB', 'Pred_FI', 'Pred_LNS', 'Pred_SUI', 'Pred_Ericsson', 'Pred_Egli', 'Pred_FSPL']

metrics_glob = {}
for n, c in zip(nombres, cols_res):
    metrics_glob[n] = calc_metrics(test_df[TARGET], test_df[c])

# ==============================================================================
# 5. TABLAS Y GRÁFICAS GLOBALES (ESTILO UNIFICADO)
# ==============================================================================
print("📊 Generando Reportes Globales...")

def save_table(data, cols, title, fname):
    plt.figure(figsize=(8, len(data)*0.5 + 1.2))
    ax = plt.gca(); ax.axis('off')
    t = ax.table(cellText=data, colLabels=cols, loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(11); t.scale(1, 1.8)
    for (i, j), cell in t.get_celld().items():
        if i == 0:
            cell.set_facecolor(COLOR_XGB) # Carmesí
            cell.set_text_props(color='white', weight='bold')
        else:
            cell.set_facecolor('#f8f9fa')
            cell.set_edgecolor('#dddddd')
    plt.title(title, fontsize=14, fontweight='bold', y=0.98)
    plt.savefig(os.path.join(DIR_SALIDA, fname), dpi=300, bbox_inches='tight'); plt.close()

# 1. Tabla XGB Global
xgb_metrics = [[k, f"{v:.4f}"] for k, v in metrics_glob['XGBoost'].items()]
save_table(xgb_metrics, ['Métrica', 'Valor'], 'Desempeño Global XGBoost (Uplink)', '1_Tabla_Global_XGB.png')

# 1B. Barras Solo XGB
plt.figure(figsize=(8, 6))
m_xgb = metrics_glob['XGBoost']
bars = plt.bar(list(m_xgb.keys()), list(m_xgb.values()), color=COLOR_XGB, edgecolor='black', width=0.6)
plt.bar_label(bars, fmt='%.2f', padding=3, fontweight='bold')
plt.title('Métricas Globales XGBoost (Uplink)', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '1B_Barras_SoloXGB_Global.png'), dpi=300); plt.close()

# 2. Tabla Comparativa
comp_data = []
for n in nombres:
    v = metrics_glob[n]
    comp_data.append([n, f"{v['RMSE']:.2f}", f"{v['MSE']:.2f}", f"{v['MAE']:.2f}", f"{v['R2']:.3f}"])
save_table(comp_data, ['Modelo', 'RMSE', 'MSE', 'MAE', 'R2'], 'Comparativa Global de Modelos', '2_Tabla_Global_Comparativa.png')

# 3. Barras Comparativas
df_m = pd.DataFrame(metrics_glob).T
for met in ['RMSE', 'MSE', 'MAE', 'R2']:
    plt.figure(figsize=(10, 6))
    df_s = df_m.sort_values(met, ascending=(met!='R2'))
    # Colores: XGB Carmesí, Empíricos Gris
    colors = [COLOR_XGB if x == 'XGBoost' else '#95a5a6' for x in df_s.index]
    bars = plt.bar(df_s.index, df_s[met], color=colors, edgecolor='black', alpha=0.9)
    plt.bar_label(bars, fmt='%.2f', fontsize=10, fontweight='bold')
    plt.title(f'Comparativa de Modelos - {met}', fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_SALIDA, f'3_Barras_Comp_{met}.png'), dpi=300); plt.close()

# 4. Curvas Comparativas
plt.figure(figsize=(12, 7))
plt.scatter(test_df['Distance'], test_df[TARGET], c='gray', alpha=0.2, s=15, label='Medición Real')

X_s = np.linspace(test_df['Distance'].min(), test_df['Distance'].max(), 300)
d_km_s = X_s/1000.0

plt.plot(X_s, 32.44+20*np.log10(f)+20*np.log10(d_km_s), 'k:', lw=1.5, label='FSPL')
plt.plot(X_s, intercept_fi+slope_fi*(10*np.log10(X_s)), color='#e67e22', lw=2, label='FI')
plt.plot(X_s, fspl_d0+n_opt*(10*np.log10(d_km_s)), color='#2ecc71', lw=2, label='LNS')

# Tendencia XGB
test_df['Bin'] = (np.round(test_df['Distance']/100)*100).astype(int)
xgb_trend = test_df.groupby('Bin')['Pred_XGB'].mean().sort_index()
plt.plot(xgb_trend.index, xgb_trend.values, color=COLOR_XGB, lw=3.5, label='XGBoost (Tendencia)')

plt.title('Comparativa de Propagación Global (Uplink)', fontweight='bold')
plt.xlabel('Distancia (m)', fontweight='bold'); plt.ylabel('Path Loss (dB)', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '4_Curvas_Comparativas.png'), dpi=300); plt.close()

# 5. HISTOGRAMA DE ERRORES (GLOBAL) - ESTILO UNIFICADO
residuos_global = test_df[TARGET] - test_df['Pred_XGB']

plt.figure(figsize=(10, 6))
sns.histplot(residuos_global, kde=True, color=COLOR_XGB, bins=40, line_kws={'linewidth': 2}, alpha=0.6)
plt.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Ideal (0 dB)')
plt.axvline(residuos_global.mean(), color='red', linestyle=':', linewidth=1.5,
            label=f'Media: {residuos_global.mean():.2f} dB')

plt.title(f'Distribución de Errores Globales - XGBoost\n(Std Dev: {residuos_global.std():.2f} dB)',
          fontsize=14, fontweight='bold')
plt.xlabel('Error (Real - Predicho) [dB]', fontweight='bold')
plt.ylabel('Frecuencia', fontweight='bold')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '5_Histograma_Errores_Global.png'), dpi=300, bbox_inches='tight')
plt.close()

# 6. Scatter Global XGB (para tener la referencia)
plt.figure(figsize=(10, 6))
plt.scatter(test_df['Distance'], test_df[TARGET], c='gray', alpha=0.5, label='Real')
plt.scatter(test_df['Distance'], test_df['Pred_XGB'], c=COLOR_XGB, marker='x', alpha=0.9, label='XGBoost')
xr = np.linspace(test_df['Distance'].min(), test_df['Distance'].max(), 100)
plt.plot(xr, 32.44 + 20*np.log10(f) + 20*np.log10(xr/1000), 'k--', label='FSPL')
plt.title('XGBoost vs Real (Global)', fontweight='bold')
plt.legend(loc='lower right'); plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '6_Scatter_Global.png'), dpi=300); plt.close()

# ==============================================================================
# 6. ANÁLISIS POR CONFIGURACIÓN (LOOP)
# ==============================================================================
print("📂 Generando Análisis por Configuración...")
DIR_CFG = os.path.join(DIR_SALIDA, 'Por_Configuracion')
if not os.path.exists(DIR_CFG): os.makedirs(DIR_CFG)

config_summary = []

for (sf, pwr), grp in test_df.groupby([col_sf, 'Power_UpLink']):
    if len(grp) < 10: continue

    nm = f"SF{sf}_P{pwr}"
    m_loc = {}
    for n, c in zip(nombres, cols_res):
        m_loc[n] = calc_metrics(grp[TARGET], grp[c])

    config_summary.append([nm, m_loc['XGBoost']['RMSE'], m_loc['XGBoost']['R2'], len(grp)])

    # 1. Tabla XGB Local
    d_xgb_loc = [[k, f"{v:.4f}"] for k, v in m_loc['XGBoost'].items()]
    save_table(d_xgb_loc, ['Métrica', 'Valor'], f'Métricas XGB - {nm}', os.path.join(DIR_CFG, f'Tabla_SoloXGB_{nm}.png'))

    # 2. Barras XGB Local
    plt.figure(figsize=(6, 4))
    vals = list(m_loc['XGBoost'].values())
    keys = list(m_loc['XGBoost'].keys())
    bars = plt.bar(keys, vals, color=COLOR_XGB, edgecolor='black')
    plt.bar_label(bars, fmt='%.2f', fontweight='bold')
    plt.title(f'Desempeño XGB - {nm}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_CFG, f'Barras_SoloXGB_{nm}.png'), dpi=300); plt.close()

    # 3. Barras Comparativas Local (RMSE)
    df_loc = pd.DataFrame(m_loc).T
    plt.figure(figsize=(10, 5))
    df_s = df_loc.sort_values('RMSE')
    colors = [COLOR_XGB if x == 'XGBoost' else '#95a5a6' for x in df_s.index]
    bars = plt.bar(df_s.index, df_s['RMSE'], color=colors, edgecolor='black')
    plt.bar_label(bars, fmt='%.2f', fontweight='bold')
    plt.title(f'Comparativa RMSE - {nm}', fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_CFG, f'Barras_Comp_RMSE_{nm}.png'), dpi=300); plt.close()

    # 4. Scatter Local (CON FSPL)
    plt.figure(figsize=(10, 6))
    plt.scatter(grp['Distance'], grp[TARGET], c='gray', alpha=0.5, label='Real')
    plt.scatter(grp['Distance'], grp['Pred_XGB'], c=COLOR_XGB, marker='x', alpha=0.9, label='Pred. XGB')

    # Línea FSPL referencia
    xr = np.linspace(grp['Distance'].min(), grp['Distance'].max(), 100)
    plt.plot(xr, 32.44+20*np.log10(f)+20*np.log10(xr/1000), 'k--', alpha=0.5, label='FSPL')

    plt.title(f'Predicción vs Real - {nm}', fontweight='bold')
    plt.xlabel('Distancia (m)'); plt.ylabel('Path Loss (dB)')
    plt.legend(loc='lower right'); plt.grid(True, linestyle='--', alpha=0.5)
    plt.savefig(os.path.join(DIR_CFG, f"Scatter_{nm}.png"), dpi=300, bbox_inches='tight'); plt.close()

    # 5. Boxplot Local
    plt.figure(figsize=(12, 6))
    bins = range(int(grp['Distance'].min()), int(grp['Distance'].max()) + BIN_SIZE + 1, BIN_SIZE)
    grp_copy = grp.copy()
    grp_copy['Bin'] = pd.cut(grp_copy['Distance'], bins=bins)

    if not grp_copy['Bin'].isnull().all():
        sns.boxplot(x='Bin', y='Pred_XGB', data=grp_copy, color=COLOR_XGB)
        plt.title(f'Distribución XGB por Distancia - {nm}', fontweight='bold')
        plt.xticks(rotation=45)
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.savefig(os.path.join(DIR_CFG, f"Boxplot_{nm}.png"), dpi=300); plt.close()

# Guardar resumen CSV
pd.DataFrame(config_summary, columns=['Config', 'RMSE', 'R2', 'Muestras']).to_csv(os.path.join(DIR_SALIDA, 'Resumen_Configs.csv'), index=False)

print(f"\n✅ TEST XGBOOST UPLINK GENERAL COMPLETADO.")
print(f"   📂 Resultados en: {DIR_SALIDA}")

# **TEST DOWNLINK**

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import linregress
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import scipy.stats as stats

# ==============================================================================
# 0. ESTILO DE TESIS (CONFIGURACIÓN MAESTRA)
# ==============================================================================
def configurar_estilo_tesis():
    sns.set_theme(style="whitegrid", context="paper")
    plt.rcParams.update({
        'font.family': 'serif',
        'axes.labelsize': 11,
        'axes.titlesize': 12,
        'xtick.labelsize': 10,
        'ytick.labelsize': 10,
        'legend.fontsize': 10,
        'figure.titlesize': 14,
        'grid.alpha': 0.3,
        'grid.linestyle': '--',
        'lines.linewidth': 2,
    })

# Color Oficial XGBoost (Carmesí)
COLOR_XGB = '#900C3F'

configurar_estilo_tesis()

# ==============================================================================
# 1. CONFIGURACIÓN (RUTAS DOWNLINK)
# ==============================================================================
DIR_DATA   = '/content/drive/MyDrive/TESIS 100%/Datos_Entrenamiento_Finales/DownLink/'
DIR_MODELO = '/content/drive/MyDrive/TESIS 100%/XGBoost/Downlink/Entrenamiento/'
DIR_SALIDA = '/content/drive/MyDrive/TESIS 100%/XGBoost/Downlink/Prueba/Test_Final_Uniforme/'

if not os.path.exists(DIR_SALIDA): os.makedirs(DIR_SALIDA)

# Constantes Físicas
FREQ_MHZ = 915.0
H_GATEWAY_EMPIRICOS = 30.0
H_NODO_EMPIRICOS = 1.5
MIN_DISTANCIA = 200
BIN_SIZE = 200

print("🚀 INICIANDO TEST GENERAL DOWNLINK - XGBOOST (ROBUSTO)...")

# ==============================================================================
# 2. CARGA DE DATOS Y MODELO
# ==============================================================================
try:
    test_df = pd.read_csv(os.path.join(DIR_DATA, 'Test_DownLink.csv'))

    # Carga Modelo
    ruta_modelo = os.path.join(DIR_MODELO, 'Modelo_XGB_RL_DownLink.pkl') # Prioridad al RL
    if not os.path.exists(ruta_modelo):
        ruta_modelo = os.path.join(DIR_MODELO, 'Modelo_XGBoost_Downlink.pkl') # Fallback 1
    if not os.path.exists(ruta_modelo):
        # Busqueda generica
        posibles = [f for f in os.listdir(DIR_MODELO) if f.endswith('.pkl')]
        if posibles: ruta_modelo = os.path.join(DIR_MODELO, posibles[0])

    if not os.path.exists(ruta_modelo):
         raise FileNotFoundError(f"No se encuentra el modelo en: {DIR_MODELO}")

    xgb_model = joblib.load(ruta_modelo)
    print(f"✅ Modelo cargado: {os.path.basename(ruta_modelo)}")

except Exception as e:
    print(f"❌ Error Fatal Carga: {e}"); exit()

# ==============================================================================
# 3. PREPROCESAMIENTO Y ALINEACIÓN (CRÍTICO)
# ==============================================================================

# --- A. AJUSTE DE POTENCIA (CONSTANTE 26 dBm) ---
# Primero, ver si existe alguna columna de potencia en el CSV
col_pwr_csv = next((c for c in ['Power_DownLink', 'Power_Downlink', 'Power'] if c in test_df.columns), None)
if col_pwr_csv:
    test_df[col_pwr_csv] = 26.0 # Forzamos el valor
else:
    test_df['Power_DownLink'] = 26.0 # Creamos si no existe

# --- B. DETECCIÓN DE TARGET ---
TARGET = next((c for c in ['Path_Loss_DownLink', 'Path_Loss_Downlink', 'Path_Loss'] if c in test_df.columns), None)
if not TARGET: print("❌ Error Fatal: No se encontró Target."); exit()

# --- C. ALINEACIÓN CON EL MODELO (FEATURE NAMES) ---
if hasattr(xgb_model, 'feature_names_in_'):
    features_esperadas = list(xgb_model.feature_names_in_)
    print(f"ℹ️ El modelo espera: {features_esperadas}")

    # 1. Alineación SF
    col_sf_csv = next((c for c in ['SpreedFactor_DownLink', 'SpreedFactor_UpLink', 'SpreedFactor', 'SF'] if c in test_df.columns), None)
    col_sf_model = next((f for f in features_esperadas if 'SpreedFactor' in f or 'SF' in f), None)

    if col_sf_model and col_sf_csv:
        if col_sf_csv != col_sf_model:
            print(f"   🔄 Renombrando '{col_sf_csv}' -> '{col_sf_model}'")
            test_df.rename(columns={col_sf_csv: col_sf_model}, inplace=True)
        col_sf_actual = col_sf_model
    else:
        print("⚠️ Advertencia: No se pudo alinear SF automáticamente.")
        col_sf_actual = col_sf_csv

    # 2. Alineación Power
    col_pwr_model = next((f for f in features_esperadas if 'Power' in f), None)
    # Buscamos la columna de potencia que tengamos actualmente en el DF
    col_pwr_current = next((c for c in test_df.columns if 'Power' in c), None)

    if col_pwr_model and col_pwr_current:
        if col_pwr_current != col_pwr_model:
            print(f"   🔄 Renombrando '{col_pwr_current}' -> '{col_pwr_model}'")
            test_df.rename(columns={col_pwr_current: col_pwr_model}, inplace=True)
            # Asegurar valor 26
            test_df[col_pwr_model] = 26.0

    FEATURES = features_esperadas
else:
    # Fallback manual
    FEATURES = ['Distance', 'Gateway_Altitude', 'EndDevice_Altitude', 'SpreedFactor_DownLink', 'Terreno', 'Power_DownLink']
    col_sf_actual = 'SpreedFactor_DownLink'

# Validación Final
missing = [c for c in FEATURES if c not in test_df.columns]
if missing:
    print(f"❌ Faltan columnas requeridas por el modelo: {missing}"); exit()

# Filtro Distancia
test_df = test_df[test_df['Distance'] >= MIN_DISTANCIA].copy()

# ==============================================================================
# 4. CÁLCULO DE PREDICCIONES
# ==============================================================================
print("📐 Calculando predicciones...")

# A. XGBoost
try:
    test_df['Pred_XGB'] = xgb_model.predict(test_df[FEATURES])
except Exception as e:
    print(f"❌ Error predicción XGB: {e}"); exit()

# B. Empíricos
d_km = test_df['Distance'] / 1000.0; d_m = test_df['Distance']
h_b = H_GATEWAY_EMPIRICOS; h_m = H_NODO_EMPIRICOS; f = FREQ_MHZ

test_df['Pred_FSPL'] = 32.44 + 20*np.log10(f) + 20*np.log10(d_km)
g_f = 44.49*np.log10(f) - 4.78*(np.log10(f))**2
test_df['Pred_Ericsson'] = (36.2 + 30.2*np.log10(d_km) - 12.0*np.log10(h_b) + 0.1*np.log10(h_b)*np.log10(d_km) - 3.2*(np.log10(11.75*h_m))**2 + g_f)
lam = 300.0 / f
A_sui = 20 * np.log10((4 * np.pi * 100) / lam)
gamma_sui = 4.0 - 0.0065 * h_b + 17.1 / h_b
test_df['Pred_SUI'] = A_sui + 10 * gamma_sui * np.log10(d_m / 100.0) + 6.0 * np.log10(f / 2000.0) - 10.8 * np.log10(h_m / 2.0)
test_df['Pred_Egli'] = 88.0 + 20*np.log10(f) + 40*np.log10(d_km) - 20*np.log10(h_b) - 10*np.log10(h_m)

# FI y LNS
x_log = 10 * np.log10(d_m)
slope_fi, intercept_fi, _, _, _ = linregress(x_log, test_df[TARGET])
test_df['Pred_FI'] = intercept_fi + slope_fi * x_log

fspl_d0 = 32.44 + 20*np.log10(f)
y_diff = test_df[TARGET] - fspl_d0
x_term = 10 * np.log10(d_km)
n_opt = np.sum(x_term * y_diff) / np.sum(x_term**2)
test_df['Pred_LNS'] = fspl_d0 + n_opt * x_term

# ==============================================================================
# 5. MÉTRICAS GLOBALES
# ==============================================================================
def calc_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {'RMSE': np.sqrt(mse), 'MSE': mse, 'MAE': mean_absolute_error(y_true, y_pred), 'R2': r2_score(y_true, y_pred)}

nombres = ['XGBoost', 'Floating Intercept', 'Log-Normal Shad.', 'SUI', 'Ericsson', 'Egli', 'FSPL']
cols_res = ['Pred_XGB', 'Pred_FI', 'Pred_LNS', 'Pred_SUI', 'Pred_Ericsson', 'Pred_Egli', 'Pred_FSPL']

metrics_glob = {}
for n, c in zip(nombres, cols_res):
    metrics_glob[n] = calc_metrics(test_df[TARGET], test_df[c])

# ==============================================================================
# 6. TABLAS Y GRÁFICAS GLOBALES
# ==============================================================================
print("📊 Generando Reportes Globales...")

def save_table(data, cols, title, fname):
    plt.figure(figsize=(8, len(data)*0.5 + 1.2))
    ax = plt.gca(); ax.axis('off')
    t = ax.table(cellText=data, colLabels=cols, loc='center', cellLoc='center')
    t.auto_set_font_size(False); t.set_fontsize(11); t.scale(1, 1.8)
    for (i, j), cell in t.get_celld().items():
        if i == 0:
            cell.set_facecolor(COLOR_XGB) # Carmesí
            cell.set_text_props(color='white', weight='bold')
        else:
            cell.set_facecolor('#f8f9fa')
            cell.set_edgecolor('#dddddd')
    plt.title(title, fontsize=14, fontweight='bold', y=0.98)
    plt.savefig(os.path.join(DIR_SALIDA, fname), dpi=300, bbox_inches='tight'); plt.close()

# 1. Tabla XGB Global
xgb_metrics = [[k, f"{v:.4f}"] for k, v in metrics_glob['XGBoost'].items()]
save_table(xgb_metrics, ['Métrica', 'Valor'], 'Desempeño Global XGBoost (Downlink)', '1_Tabla_Global_XGB.png')

# 1B. Barras Solo XGB
plt.figure(figsize=(8, 6))
m_xgb = metrics_glob['XGBoost']
bars = plt.bar(list(m_xgb.keys()), list(m_xgb.values()), color=COLOR_XGB, edgecolor='black', width=0.6)
plt.bar_label(bars, fmt='%.2f', padding=3, fontweight='bold')
plt.title('Métricas Globales XGBoost (Downlink)', fontweight='bold')
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '1B_Barras_SoloXGB_Global.png'), dpi=300); plt.close()

# 2. Tabla Comparativa
comp_data = []
for n in nombres:
    v = metrics_glob[n]
    comp_data.append([n, f"{v['RMSE']:.2f}", f"{v['MSE']:.2f}", f"{v['MAE']:.2f}", f"{v['R2']:.3f}"])
save_table(comp_data, ['Modelo', 'RMSE', 'MSE', 'MAE', 'R2'], 'Comparativa Global de Modelos (DL)', '2_Tabla_Global_Comparativa.png')

# 3. Barras Comparativas
df_m = pd.DataFrame(metrics_glob).T
for met in ['RMSE', 'MSE', 'MAE', 'R2']:
    plt.figure(figsize=(10, 6))
    df_s = df_m.sort_values(met, ascending=(met!='R2'))
    colors = [COLOR_XGB if x == 'XGBoost' else '#95a5a6' for x in df_s.index]
    bars = plt.bar(df_s.index, df_s[met], color=colors, edgecolor='black', alpha=0.9)
    plt.bar_label(bars, fmt='%.2f', fontsize=10, fontweight='bold')
    plt.title(f'Comparativa de Modelos DL - {met}', fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_SALIDA, f'3_Barras_Comp_{met}.png'), dpi=300); plt.close()

# 4. Curvas Comparativas
plt.figure(figsize=(12, 7))
plt.scatter(test_df['Distance'], test_df[TARGET], c='gray', alpha=0.2, s=15, label='Medición Real')

X_s = np.linspace(test_df['Distance'].min(), test_df['Distance'].max(), 300)
d_km_s = X_s/1000.0

plt.plot(X_s, 32.44+20*np.log10(f)+20*np.log10(d_km_s), 'k:', lw=1.5, label='FSPL')
plt.plot(X_s, intercept_fi+slope_fi*(10*np.log10(X_s)), color='#e67e22', lw=2, label='FI')
plt.plot(X_s, fspl_d0+n_opt*(10*np.log10(d_km_s)), color='#2ecc71', lw=2, label='LNS')

# Tendencia XGB
test_df['Bin'] = (np.round(test_df['Distance']/100)*100).astype(int)
xgb_trend = test_df.groupby('Bin')['Pred_XGB'].mean().sort_index()
plt.plot(xgb_trend.index, xgb_trend.values, color=COLOR_XGB, lw=3.5, label='XGBoost (Tendencia)')

plt.title('Comparativa de Propagación Global (Downlink)', fontweight='bold')
plt.xlabel('Distancia (m)', fontweight='bold'); plt.ylabel('Path Loss (dB)', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '4_Curvas_Comparativas.png'), dpi=300); plt.close()

# 5. HISTOGRAMA DE ERRORES (GLOBAL) - ESTILO UNIFICADO
residuos_global = test_df[TARGET] - test_df['Pred_XGB']

plt.figure(figsize=(10, 6))
sns.histplot(residuos_global, kde=True, color=COLOR_XGB, bins=40, line_kws={'linewidth': 2}, alpha=0.6)
plt.axvline(0, color='black', linestyle='--', linewidth=1.5, label='Ideal (0 dB)')
plt.axvline(residuos_global.mean(), color='red', linestyle=':', linewidth=1.5,
            label=f'Media: {residuos_global.mean():.2f} dB')

plt.title(f'Distribución de Errores Globales DL - XGBoost\n(Std Dev: {residuos_global.std():.2f} dB)',
          fontsize=14, fontweight='bold')
plt.xlabel('Error (Real - Predicho) [dB]', fontweight='bold')
plt.ylabel('Frecuencia', fontweight='bold')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '5_Histograma_Errores_Global.png'), dpi=300, bbox_inches='tight')
plt.close()

# 6. SCATTER REAL VS PREDICHO GLOBAL (Con FSPL)
plt.figure(figsize=(10, 6))
plt.scatter(test_df['Distance'], test_df[TARGET], c='gray', alpha=0.5, label='Real')
plt.scatter(test_df['Distance'], test_df['Pred_XGB'], c=COLOR_XGB, marker='x', alpha=0.9, label='XGBoost')
xr = np.linspace(test_df['Distance'].min(), test_df['Distance'].max(), 100)
plt.plot(xr, 32.44 + 20*np.log10(f) + 20*np.log10(xr/1000), 'k--', alpha=0.6, label='FSPL')
plt.title('XGBoost vs Real (Global DL)', fontweight='bold')
plt.legend(loc='lower right'); plt.grid(True, linestyle='--', alpha=0.5)
plt.savefig(os.path.join(DIR_SALIDA, '6_Scatter_Global.png'), dpi=300, bbox_inches='tight'); plt.close()

# ==============================================================================
# 7. ANÁLISIS POR CONFIGURACIÓN (LOOP)
# ==============================================================================
print("📂 Generando Análisis por Configuración...")
DIR_CFG = os.path.join(DIR_SALIDA, 'Por_Configuracion')
if not os.path.exists(DIR_CFG): os.makedirs(DIR_CFG)

config_summary = []

# Agrupamos por la columna de SF normalizada y la potencia (o fija 26)
col_pwr_final = next((c for c in test_df.columns if 'Power' in c), None)

for (sf, pwr), grp in test_df.groupby([col_sf_actual, col_pwr_final]):
    if len(grp) < 10: continue

    nm = f"SF{sf}_P{pwr}"
    m_loc = {}
    for n, c in zip(nombres, cols_res):
        m_loc[n] = calc_metrics(grp[TARGET], grp[c])

    config_summary.append([nm, m_loc['XGBoost']['RMSE'], m_loc['XGBoost']['R2'], len(grp)])

    # 1. Tabla XGB Local
    d_xgb_loc = [[k, f"{v:.4f}"] for k, v in m_loc['XGBoost'].items()]
    save_table(d_xgb_loc, ['Métrica', 'Valor'], f'Métricas XGB DL - {nm}', os.path.join(DIR_CFG, f'Tabla_SoloXGB_{nm}.png'))

    # 2. Barras XGB Local
    plt.figure(figsize=(6, 4))
    vals = list(m_loc['XGBoost'].values())
    keys = list(m_loc['XGBoost'].keys())
    bars = plt.bar(keys, vals, color=COLOR_XGB, edgecolor='black')
    plt.bar_label(bars, fmt='%.2f', fontweight='bold')
    plt.title(f'Desempeño XGB - {nm}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_CFG, f'Barras_SoloXGB_{nm}.png'), dpi=300); plt.close()

    # 3. Barras Comparativas Local (RMSE)
    df_loc = pd.DataFrame(m_loc).T
    plt.figure(figsize=(10, 5))
    df_s = df_loc.sort_values('RMSE')
    colors = [COLOR_XGB if x == 'XGBoost' else '#95a5a6' for x in df_s.index]
    bars = plt.bar(df_s.index, df_s['RMSE'], color=colors, edgecolor='black')
    plt.bar_label(bars, fmt='%.2f', fontweight='bold')
    plt.title(f'Comparativa RMSE DL - {nm}', fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(os.path.join(DIR_CFG, f'Barras_Comp_RMSE_{nm}.png'), dpi=300); plt.close()

    # 4. Scatter Local (CON FSPL)
    plt.figure(figsize=(10, 6))
    plt.scatter(grp['Distance'], grp[TARGET], c='gray', alpha=0.5, label='Real')
    plt.scatter(grp['Distance'], grp['Pred_XGB'], c=COLOR_XGB, marker='x', alpha=0.9, label='Pred. XGB')

    xr = np.linspace(grp['Distance'].min(), grp['Distance'].max(), 100)
    plt.plot(xr, 32.44+20*np.log10(f)+20*np.log10(xr/1000), 'k--', alpha=0.6, label='FSPL')

    plt.title(f'Predicción vs Real DL - {nm}', fontweight='bold')
    plt.xlabel('Distancia (m)'); plt.ylabel('Path Loss (dB)')
    plt.legend(loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.savefig(os.path.join(DIR_CFG, f"Scatter_{nm}.png"), dpi=300, bbox_inches='tight'); plt.close()

    # 5. Boxplot Local
    plt.figure(figsize=(12, 6))
    bins = range(int(grp['Distance'].min()), int(grp['Distance'].max()) + BIN_SIZE + 1, BIN_SIZE)
    grp_copy = grp.copy()
    grp_copy['Bin'] = pd.cut(grp_copy['Distance'], bins=bins)

    if not grp_copy['Bin'].isnull().all():
        sns.boxplot(x='Bin', y='Pred_XGB', data=grp_copy, color=COLOR_XGB)
        plt.title(f'Distribución XGB por Distancia - {nm}', fontweight='bold')
        plt.xticks(rotation=45)
        plt.grid(axis='y', linestyle='--', alpha=0.5)
        plt.tight_layout()
        plt.savefig(os.path.join(DIR_CFG, f"Boxplot_{nm}.png"), dpi=300); plt.close()

# Guardar resumen CSV
pd.DataFrame(config_summary, columns=['Config', 'RMSE', 'R2', 'Muestras']).to_csv(os.path.join(DIR_SALIDA, 'Resumen_Configs.csv'), index=False)

print(f"\n✅ TEST GENERAL XGBOOST DOWNLINK COMPLETADO.")
print(f"   📂 Resultados en: {DIR_SALIDA}")